# SHAP-Gesamtvergleich über alle Frameworks und Aufgaben



In [ ]:
import os
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D


In [ ]:
# ================================================================
# CONFIG
# ================================================================
HOST_BASE = r"E:\StudiumMasterarbeit"
SHAP_RESULTS_DIR = os.path.join(HOST_BASE, "shap_results")

FRAMEWORKS = ["auto-sklearn", "auto-pytorch", "autogluon"]
FRAMEWORK_DIR_NAMES = {
    "auto-sklearn": "auto-sklearn",
    "auto-pytorch": "auto-pytorch",
    "autogluon": "autogluon",
}

TOP_N_FEATURES = 10

CATEGORY_ORDER = [
    "Classification",
    "Anthropometric regression",
    "Force regression",
    "Other",
]

CLASSIFICATION_TASKS = {
    "sex_classification",
    "age_bracket_classification",
    "height_bracket_classification",
}

ANTHROPOMETRIC_TASKS = {
    "age_regression",
    "height_regression",
}

def categorize_task(task_name):
    if task_name in CLASSIFICATION_TASKS:
        return "Classification"
    if task_name in ANTHROPOMETRIC_TASKS:
        return "Anthropometric regression"
    if task_name.startswith("predict_F_"):
        return "Force regression"
    return "Other"

print(f"SHAP-Ergebnisse: {SHAP_RESULTS_DIR}")


In [ ]:
# ================================================================
# 1. VORHANDENE SHAP-ERGEBNISSE FINDEN
# ================================================================
results = []

for framework, framework_dir in FRAMEWORK_DIR_NAMES.items():
    framework_path = os.path.join(SHAP_RESULTS_DIR, framework_dir)

    if not os.path.isdir(framework_path):
        continue

    for task_name in sorted(os.listdir(framework_path)):
        task_path = os.path.join(framework_path, task_name)
        shap_path = os.path.join(task_path, "shap_values.npy")
        feature_path = os.path.join(task_path, "feature_names.json")

        if not os.path.isdir(task_path):
            continue
        if not os.path.isfile(shap_path):
            continue
        if not os.path.isfile(feature_path):
            continue

        results.append({
            "framework": framework,
            "task": task_name,
            "category": categorize_task(task_name),
            "path": task_path,
        })

results_index = pd.DataFrame(results)

print(f"Gefundene Framework × Aufgabe-Kombinationen: {len(results_index)}")

if results_index.empty:
    print("Keine Ergebnisse gefunden. Bitte HOST_BASE und SHAP_RESULTS_DIR prüfen.")
else:
    overview = results_index.pivot_table(
        index="task",
        columns="framework",
        values="path",
        aggfunc=lambda _: "x",
        fill_value="-",
    )
    overview = overview.reindex(
        sorted(overview.index, key=lambda t: (categorize_task(t), t))
    )

    print("\nÜbersicht:")
    print(overview.to_string())

    print("\nAbdeckung pro Kategorie:")
    print(
        results_index
        .groupby(["category", "framework"])
        .size()
        .unstack(fill_value=0)
        .to_string()
    )


In [ ]:
# ================================================================
# 2. ALLE ERGEBNISSE LADEN UND GEMEINSAME TABELLEN AUFBAUEN
# ================================================================
importance_rows = []
efficiency_rows = []

for _, result in results_index.iterrows():
    path = result["path"]

    shap_values = np.load(os.path.join(path, "shap_values.npy"))

    with open(os.path.join(path, "feature_names.json"), encoding="utf-8") as f:
        feature_names = json.load(f)

    mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
    ranks = pd.Series(-mean_abs_shap).rank(method="min").astype(int)

    for feature, importance, rank in zip(feature_names, mean_abs_shap, ranks):
        importance_rows.append({
            "framework": result["framework"],
            "task": result["task"],
            "category": result["category"],
            "feature": feature,
            "mean_abs_shap": importance,
            "rank": rank,
        })

    base_path = os.path.join(path, "base_values.npy")
    prediction_path = os.path.join(path, "y_pred_explained.npy")

    if os.path.isfile(base_path) and os.path.isfile(prediction_path):
        base_values = np.load(base_path)
        y_pred = np.load(prediction_path)

        reconstruction = shap_values.sum(axis=1) + base_values
        efficiency_gap = np.mean(np.abs(y_pred - reconstruction))

        efficiency_rows.append({
            "framework": result["framework"],
            "task": result["task"],
            "category": result["category"],
            "mean_efficiency_gap": efficiency_gap,
        })

importance_df = pd.DataFrame(importance_rows)
efficiency_df = pd.DataFrame(efficiency_rows)

print(f"importance_df: {len(importance_df)} Zeilen")

if not efficiency_df.empty:
    print(f"efficiency_df: {len(efficiency_df)} Zeilen")
    n_bad = (efficiency_df["mean_efficiency_gap"] > 1e-4).sum()
    print(f"Auffällige Efficiency-Gaps (>1e-4): {n_bad}")


In [ ]:
# ================================================================
# 3. VERGLEICHSTABELLEN
# ================================================================

# Mittlere SHAP-Größenordnung pro Framework und Aufgaben-Kategorie
magnitude_summary = (
    importance_df
    .groupby(["category", "framework"])["mean_abs_shap"]
    .agg(["mean", "std", "max", "count"])
    .rename(columns={
        "mean": "mean_abs_shap",
        "std": "std_abs_shap",
        "max": "max_abs_shap",
        "count": "n_feature_rows",
    })
    .reset_index()
)

magnitude_summary = magnitude_summary[
    magnitude_summary["category"].isin(CATEGORY_ORDER)
]

print("=== Mittlere SHAP-Größenordnung ===")
print(magnitude_summary.to_string(index=False))


# Top-N Merkmale pro Aufgabe
top_feature_tables = {}

for task_name in sorted(
    importance_df["task"].unique(),
    key=lambda t: (categorize_task(t), t)
):
    task_data = importance_df[importance_df["task"] == task_name]

    pivot = task_data.pivot_table(
        index="feature",
        columns="framework",
        values="mean_abs_shap",
        fill_value=0.0,
    )

    pivot["max_importance"] = pivot.max(axis=1)
    pivot = pivot.sort_values("max_importance", ascending=False)
    pivot = pivot.drop(columns="max_importance").head(TOP_N_FEATURES)

    top_feature_tables[task_name] = pivot

    print(
        f"\n=== Top {TOP_N_FEATURES} Merkmale: "
        f"{task_name} ({categorize_task(task_name)}) ==="
    )
    print(pivot.round(5).to_string())


# Spearman-Rangkorrelation zwischen Frameworks
correlation_rows = []

for task_name in sorted(importance_df["task"].unique()):
    task_data = importance_df[importance_df["task"] == task_name]

    pivot = task_data.pivot_table(
        index="feature",
        columns="framework",
        values="mean_abs_shap",
        fill_value=0.0,
    )

    available_frameworks = [
        fw for fw in FRAMEWORKS if fw in pivot.columns
    ]

    row = {
        "task": task_name,
        "category": categorize_task(task_name),
    }

    for i in range(len(available_frameworks)):
        for j in range(i + 1, len(available_frameworks)):
            fw_a = available_frameworks[i]
            fw_b = available_frameworks[j]
            rho, _ = spearmanr(pivot[fw_a], pivot[fw_b])
            row[f"{fw_a} vs {fw_b}"] = rho

    correlation_rows.append(row)

correlation_df = pd.DataFrame(correlation_rows).set_index("task")
corr_cols = [c for c in correlation_df.columns if "vs" in c]

print("\n=== Spearman-Rangkorrelation pro Aufgabe ===")
print(correlation_df.round(3).to_string())

print("\n=== Mittlere Rangkorrelation pro Kategorie ===")
print(
    correlation_df
    .groupby("category")[corr_cols]
    .mean()
    .round(3)
    .to_string()
)


In [ ]:
# ================================================================
# 4. VERGLEICHS-PLOTS
# ================================================================

# Plot 1: SHAP-Größenordnung pro Framework und Kategorie
categories_present = [
    category
    for category in CATEGORY_ORDER
    if category in magnitude_summary["category"].unique()
]

fig, axes = plt.subplots(
    1,
    len(categories_present),
    figsize=(5.5 * len(categories_present), 4.5),
    sharey=True,
)

if len(categories_present) == 1:
    axes = [axes]

framework_colors = {
    "auto-sklearn": "#4C72B0",
    "auto-pytorch": "#DD8452",
    "autogluon": "#55A868",
}

for ax, category in zip(axes, categories_present):
    subset = magnitude_summary[
        magnitude_summary["category"] == category
    ]

    frameworks = subset["framework"].tolist()
    means = subset["mean_abs_shap"].tolist()
    stds = subset["std_abs_shap"].tolist()

    ax.bar(
        frameworks,
        means,
        yerr=stds,
        capsize=4,
        color=[framework_colors.get(fw, "gray") for fw in frameworks],
        edgecolor="black",
        linewidth=0.8,
    )

    ax.set_title(category)
    ax.set_ylabel("Mean |SHAP value|" if ax is axes[0] else "")
    ax.set_yscale("log")
    ax.tick_params(axis="x", rotation=20)
    ax.spines[["top", "right"]].set_visible(False)

    for i, value in enumerate(means):
        label = f"{value:.4f}" if value < 1 else f"{value:.2f}"
        ax.annotate(
            label,
            (i, value),
            textcoords="offset points",
            xytext=(0, 8),
            ha="center",
            fontsize=9,
        )

y_min = min(ax.get_ylim()[0] for ax in axes)
y_max = max(ax.get_ylim()[1] for ax in axes)

for ax in axes:
    ax.set_ylim(y_min, y_max)

plt.suptitle(
    "SHAP attribution magnitude by framework and task category",
    y=1.03,
)
plt.tight_layout()
plt.savefig(
    os.path.join(
        SHAP_RESULTS_DIR,
        "_comparison_magnitude_by_category.png",
    ),
    dpi=200,
    bbox_inches="tight",
)
plt.show()


# Plot 2: Framework-Übereinstimmung
corr_plot_df = correlation_df[corr_cols].copy()
corr_plot_df = corr_plot_df.reindex(
    sorted(
        corr_plot_df.index,
        key=lambda t: (categorize_task(t), t),
    )
)

fig, ax = plt.subplots(
    figsize=(6, max(4, 0.35 * len(corr_plot_df)))
)

image = ax.imshow(
    corr_plot_df.values,
    cmap="RdYlGn",
    vmin=-1,
    vmax=1,
    aspect="auto",
)

ax.set_xticks(range(len(corr_plot_df.columns)))
ax.set_xticklabels(
    corr_plot_df.columns,
    rotation=30,
    ha="right",
)

ax.set_yticks(range(len(corr_plot_df.index)))
ax.set_yticklabels(corr_plot_df.index)

for i in range(corr_plot_df.shape[0]):
    for j in range(corr_plot_df.shape[1]):
        value = corr_plot_df.iloc[i, j]
        ax.text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=8,
        )

plt.colorbar(
    image,
    ax=ax,
    label="Spearman rank correlation",
)

ax.set_title(
    "Pairwise framework agreement on feature importance ranking"
)

plt.tight_layout()
plt.savefig(
    os.path.join(
        SHAP_RESULTS_DIR,
        "_comparison_rank_correlation_heatmap.png",
    ),
    dpi=200,
    bbox_inches="tight",
)
plt.show()


# Plot 3: Top-Merkmale einer ausgewählten Aufgabe
TASK_TO_PLOT = sorted(importance_df["task"].unique())[0]

pivot = top_feature_tables[TASK_TO_PLOT]

fig, ax = plt.subplots(
    figsize=(8, 0.4 * len(pivot) + 1.5)
)

pivot.plot(
    kind="barh",
    ax=ax,
    color=[
        framework_colors.get(framework, "gray")
        for framework in pivot.columns
    ],
)

ax.invert_yaxis()
ax.set_xlabel("Mean |SHAP value|")
ax.set_title(
    f"Top {TOP_N_FEATURES} features - "
    f"{TASK_TO_PLOT} ({categorize_task(TASK_TO_PLOT)})"
)
ax.legend(title="Framework")
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(
    os.path.join(
        SHAP_RESULTS_DIR,
        f"_comparison_top_features_{TASK_TO_PLOT}.png",
    ),
    dpi=200,
    bbox_inches="tight",
)
plt.show()


In [ ]:
# ================================================================
# 5. CSV-EXPORT
# ================================================================
export_dir = os.path.join(SHAP_RESULTS_DIR, "_comparison_exports")
os.makedirs(export_dir, exist_ok=True)

importance_df.to_csv(
    os.path.join(export_dir, "importance_long.csv"),
    index=False,
)

magnitude_summary.to_csv(
    os.path.join(export_dir, "magnitude_summary_by_category.csv"),
    index=False,
)

correlation_df.to_csv(
    os.path.join(export_dir, "rank_correlation_by_task.csv"),
)

if not efficiency_df.empty:
    efficiency_df.to_csv(
        os.path.join(export_dir, "efficiency_check.csv"),
        index=False,
    )

print(f"CSV-Exporte gespeichert unter: {export_dir}")
for filename in sorted(os.listdir(export_dir)):
    print(f"  - {filename}")


---
# Gait-Cycle-Heatmaps



In [ ]:
# ================================================================
# 6. GAIT-DATEN: CONFIG
# ================================================================
GAIT_DATA_DIR = os.path.join(HOST_BASE, "GutenbergGaitDatabase")

CURVE_FILES = {
    "F_V_PRO": "GRF_F_V_PRO_left.csv",
    "F_AP_PRO": "GRF_F_AP_PRO_left.csv",
    "F_ML_PRO": "GRF_F_ML_PRO_left.csv",
}

DIRECTIONS = ["F_V_PRO", "F_AP_PRO", "F_ML_PRO"]
PERCENTAGES = [20, 40, 60, 80]
N_POINTS = 101

GAIT_OUTPUT_DIR = os.path.join(
    SHAP_RESULTS_DIR,
    "_gait_cycle_heatmaps",
)
os.makedirs(GAIT_OUTPUT_DIR, exist_ok=True)

print(f"Gait-Daten: {GAIT_DATA_DIR}")
print(f"Gait-Ausgaben: {GAIT_OUTPUT_DIR}")


In [ ]:
gait_curves = {}
gait_importance = {}

for direction in DIRECTIONS:
    csv_path = os.path.join(GAIT_DATA_DIR, CURVE_FILES[direction])
    curve_df = pd.read_csv(csv_path)

    point_columns = {}
    pattern = re.compile(rf"^{re.escape(direction)}_(\d+)$")

    for column in curve_df.columns:
        match = pattern.match(column)
        if match:
            point_columns[int(match.group(1))] = column

    points = np.array(sorted(point_columns.keys()))
    mean_curve = np.array(
        [curve_df[point_columns[p]].mean() for p in points]
    )

    gait_curves[direction] = {
        "points": points,
        "mean_curve": mean_curve,
    }


# Task-Namen in Richtung, Zielprozent und Zielpunkt zerlegen
task_info = {}

task_pattern = re.compile(
    r"predict_(F_[A-Z]+_PRO)_(\d+)pct_F_[A-Z]+_PRO_(\d+)$"
)

for task_name in importance_df["task"].unique():
    match = task_pattern.match(task_name)

    if match:
        task_info[task_name] = {
            "direction": match.group(1),
            "percentage": int(match.group(2)),
            "target_point": int(match.group(3)),
        }


# SHAP-Importance jedes Frameworks auf die 101 Kurvenpunkte verteilen
for direction in DIRECTIONS:
    gait_importance[direction] = {}

    direction_tasks = [
        task for task, info in task_info.items()
        if info["direction"] == direction
    ]

    for framework in FRAMEWORKS:
        gait_importance[direction][framework] = {}

        for task_name in direction_tasks:
            task_data = importance_df[
                (importance_df["framework"] == framework)
                & (importance_df["task"] == task_name)
            ]

            importance_curve = np.full(N_POINTS, np.nan)

            for _, row in task_data.iterrows():
                match = re.search(
                    r"_(\d+)$",
                    str(row["feature"]),
                )

                if not match:
                    continue

                point = int(match.group(1))

                if 1 <= point <= N_POINTS:
                    importance_curve[point - 1] = row["mean_abs_shap"]

            gait_importance[direction][framework][task_name] = (
                importance_curve
            )

print("Gait-Daten und SHAP-Importances vorbereitet.")


Gait-Daten und SHAP-Importances vorbereitet.


In [ ]:
# ================================================================
# 8. GAIT-CYCLE: PER-TASK UND AGGREGIERT
# ================================================================
for direction in DIRECTIONS:
    points = gait_curves[direction]["points"]
    mean_curve = gait_curves[direction]["mean_curve"]

    # ------------------------------------------------------------
    # (a) Per-task: 20/40/60/80% × Framework
    # ------------------------------------------------------------
    task_lookup = {}

    for percentage in PERCENTAGES:
        matches = [
            task for task, info in task_info.items()
            if info["direction"] == direction
            and info["percentage"] == percentage
        ]

        if matches:
            task_lookup[percentage] = matches[0]

    all_curves = []

    for percentage, task_name in task_lookup.items():
        for framework in FRAMEWORKS:
            curve = gait_importance[direction][framework].get(task_name)

            if curve is not None:
                valid = curve[~np.isnan(curve)]
                if len(valid):
                    all_curves.append(valid)

    all_values = (
        np.concatenate(all_curves)
        if all_curves
        else np.array([0.0, 1.0])
    )

    vmin = float(np.min(all_values))
    vmax = float(np.max(all_values))

    fig, axes = plt.subplots(
        len(PERCENTAGES),
        len(FRAMEWORKS),
        figsize=(4.2 * len(FRAMEWORKS), 2.6 * len(PERCENTAGES)),
        sharex=True,
    )

    if len(PERCENTAGES) == 1:
        axes = np.array([axes])

    last_line = None

    for row_index, percentage in enumerate(PERCENTAGES):
        task_name = task_lookup.get(percentage)

        for column_index, framework in enumerate(FRAMEWORKS):
            ax = axes[row_index, column_index]

            if task_name is None:
                ax.set_visible(False)
                continue

            importance_curve = gait_importance[direction][framework][task_name]

            segments = np.stack(
                [
                    np.column_stack(
                        [points[:-1], mean_curve[:-1]]
                    ),
                    np.column_stack(
                        [points[1:], mean_curve[1:]]
                    ),
                ],
                axis=1,
            )

            segment_importance = np.nanmean(
                np.vstack(
                    [
                        importance_curve[:-1],
                        importance_curve[1:],
                    ]
                ),
                axis=0,
            )

            valid_segments = ~np.isnan(segment_importance)

            if np.any(~valid_segments):
                ax.add_collection(
                    LineCollection(
                        segments[~valid_segments],
                        colors="lightgray",
                        linewidths=2,
                        linestyles="--",
                    )
                )

            if np.any(valid_segments):
                line = LineCollection(
                    segments[valid_segments],
                    cmap="inferno",
                    norm=Normalize(vmin=vmin, vmax=vmax),
                    linewidths=2.5,
                )
                line.set_array(segment_importance[valid_segments])
                ax.add_collection(line)
                last_line = line

            ax.set_xlim(points.min(), points.max())

            padding = 0.08 * (
                np.nanmax(mean_curve)
                - np.nanmin(mean_curve)
                + 1e-9
            )

            ax.set_ylim(
                np.nanmin(mean_curve) - padding,
                np.nanmax(mean_curve) + padding,
            )

            ax.set_title(
                framework if row_index == 0 else "",
                fontsize=10,
            )
            ax.set_xlabel("% of gait cycle")
            ax.spines[["top", "right"]].set_visible(False)

            if column_index == 0:
                ax.set_ylabel(f"{percentage}% target\nForce")

    if last_line is not None:
        fig.colorbar(
            last_line,
            ax=axes,
            shrink=0.8,
            label="Mean |SHAP value|",
            pad=0.02,
        )

    fig.suptitle(
        f"SHAP importance mapped onto {direction} curve (per target task)",
        y=1.01,
    )

    fig.savefig(
        os.path.join(
            GAIT_OUTPUT_DIR,
            f"gait_heatmap_per_task_{direction}.png",
        ),
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()


    # ------------------------------------------------------------
    # (b) Aggregiert: Mittelwert über alle Zielaufgaben
    # ------------------------------------------------------------
    matching_tasks = [
        task for task, info in task_info.items()
        if info["direction"] == direction
    ]

    if not matching_tasks:
        print(
            f"Keine Aufgaben für Richtung {direction} gefunden – überspringe."
        )
        continue

    aggregated = {}

    for framework in FRAMEWORKS:
        curves = [
            gait_importance[direction][framework][task]
            for task in matching_tasks
            if task in gait_importance[direction][framework]
        ]

        aggregated[framework] = (
            np.nanmean(np.vstack(curves), axis=0)
            if curves
            else np.full(N_POINTS, np.nan)
        )

    all_values = np.concatenate([
        curve[~np.isnan(curve)]
        for curve in aggregated.values()
        if np.any(~np.isnan(curve))
    ])

    vmin = float(np.min(all_values))
    vmax = float(np.max(all_values))

    fig, axes = plt.subplots(
        1,
        len(FRAMEWORKS),
        figsize=(4.5 * len(FRAMEWORKS), 3.8),
        sharey=True,
    )

    if len(FRAMEWORKS) == 1:
        axes = [axes]

    last_line = None

    for ax, framework in zip(axes, FRAMEWORKS):
        importance_curve = aggregated[framework]

        segments = np.stack(
            [
                np.column_stack(
                    [points[:-1], mean_curve[:-1]]
                ),
                np.column_stack(
                    [points[1:], mean_curve[1:]]
                ),
            ],
            axis=1,
        )

        segment_importance = np.nanmean(
            np.vstack(
                [
                    importance_curve[:-1],
                    importance_curve[1:],
                ]
            ),
            axis=0,
        )

        valid_segments = ~np.isnan(segment_importance)

        if np.any(~valid_segments):
            ax.add_collection(
                LineCollection(
                    segments[~valid_segments],
                    colors="lightgray",
                    linewidths=2,
                    linestyles="--",
                )
            )

        if np.any(valid_segments):
            line = LineCollection(
                segments[valid_segments],
                cmap="inferno",
                norm=Normalize(vmin=vmin, vmax=vmax),
                linewidths=2.5,
            )
            line.set_array(segment_importance[valid_segments])
            ax.add_collection(line)
            last_line = line

        ax.set_xlim(points.min(), points.max())

        padding = 0.08 * (
            np.nanmax(mean_curve)
            - np.nanmin(mean_curve)
            + 1e-9
        )

        ax.set_ylim(
            np.nanmin(mean_curve) - padding,
            np.nanmax(mean_curve) + padding,
        )

        ax.set_title(framework, fontsize=10)
        ax.set_xlabel("% of gait cycle")
        ax.spines[["top", "right"]].set_visible(False)

    axes[0].set_ylabel("Force")

    if last_line is not None:
        fig.colorbar(
            last_line,
            ax=axes,
            shrink=0.85,
            label="Mean |SHAP value| (avg. over 20/40/60/80pct)",
            pad=0.02,
        )

    fig.suptitle(
        f"SHAP importance mapped onto {direction} curve "
        "(aggregated across all target tasks)",
        y=1.03,
    )

    fig.savefig(
        os.path.join(
            GAIT_OUTPUT_DIR,
            f"gait_heatmap_aggregated_{direction}.png",
        ),
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()

print(f"Alle Gait-Cycle-Heatmaps gespeichert unter: {GAIT_OUTPUT_DIR}")


---
# Overlay: alle drei Kraftrichtungen



In [ ]:
# ================================================================
# 9. OVERLAY ALLER KRAFTRICHTUNGEN
# ================================================================
LINESTYLES = {
    "F_V_PRO": "-",
    "F_AP_PRO": "--",
    "F_ML_PRO": ":",
}

overlay_data = {}

for direction in DIRECTIONS:
    points = gait_curves[direction]["points"]
    mean_curve = gait_curves[direction]["mean_curve"]

    curve_min = np.nanmin(mean_curve)
    curve_max = np.nanmax(mean_curve)

    if curve_max - curve_min < 1e-12:
        mean_curve_normalized = np.zeros_like(mean_curve)
    else:
        mean_curve_normalized = (
            (mean_curve - curve_min)
            / (curve_max - curve_min)
        )

    matching_tasks = [
        task for task, info in task_info.items()
        if info["direction"] == direction
    ]

    if not matching_tasks:
        continue

    importance_by_framework = {}

    for framework in FRAMEWORKS:
        curves = [
            gait_importance[direction][framework][task]
            for task in matching_tasks
            if task in gait_importance[direction][framework]
        ]

        if curves:
            importance_by_framework[framework] = np.nanmean(
                np.vstack(curves),
                axis=0,
            )

    overlay_data[direction] = {
        "points": points,
        "mean_curve_normalized": mean_curve_normalized,
        "importance_by_framework": importance_by_framework,
    }


all_overlay_values = []

for direction_data in overlay_data.values():
    for importance_curve in direction_data["importance_by_framework"].values():
        valid = importance_curve[~np.isnan(importance_curve)]
        if len(valid):
            all_overlay_values.append(valid)

all_overlay_values = np.concatenate(all_overlay_values)

vmin = float(np.min(all_overlay_values))
vmax = float(np.max(all_overlay_values))

fig, axes = plt.subplots(
    1,
    len(FRAMEWORKS),
    figsize=(5.5 * len(FRAMEWORKS), 4.2),
    sharey=True,
)

if len(FRAMEWORKS) == 1:
    axes = [axes]

last_line = None
normalization = Normalize(vmin=vmin, vmax=vmax)

for ax, framework in zip(axes, FRAMEWORKS):
    for direction, data in overlay_data.items():
        importance_curve = data["importance_by_framework"].get(framework)

        if importance_curve is None:
            continue

        points = data["points"]
        force_curve = data["mean_curve_normalized"]

        segments = np.stack(
            [
                np.column_stack(
                    [points[:-1], force_curve[:-1]]
                ),
                np.column_stack(
                    [points[1:], force_curve[1:]]
                ),
            ],
            axis=1,
        )

        segment_importance = np.nanmean(
            np.vstack(
                [
                    importance_curve[:-1],
                    importance_curve[1:],
                ]
            ),
            axis=0,
        )

        valid_segments = ~np.isnan(segment_importance)

        if np.any(~valid_segments):
            ax.add_collection(
                LineCollection(
                    segments[~valid_segments],
                    colors="lightgray",
                    linewidths=1.5,
                    linestyles=LINESTYLES[direction],
                )
            )

        if np.any(valid_segments):
            line = LineCollection(
                segments[valid_segments],
                cmap="inferno",
                norm=normalization,
                linewidths=2.5,
                linestyles=LINESTYLES[direction],
            )
            line.set_array(segment_importance[valid_segments])
            ax.add_collection(line)
            last_line = line

    ax.set_xlim(1, N_POINTS)
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(framework)
    ax.set_xlabel("% of gait cycle")
    ax.spines[["top", "right"]].set_visible(False)

axes[0].set_ylabel(
    "Normalized force (min-max per direction)"
)

legend_elements = [
    Line2D(
        [0],
        [0],
        color="black",
        linestyle=linestyle,
        label=direction,
    )
    for direction, linestyle in LINESTYLES.items()
]

axes[-1].legend(
    handles=legend_elements,
    loc="upper right",
    fontsize=8,
    title="Direction",
)

if last_line is not None:
    fig.colorbar(
        last_line,
        ax=axes,
        shrink=0.85,
        label="Mean |SHAP value| (aggregated)",
        pad=0.02,
    )

fig.suptitle(
    "SHAP importance across all force directions "
    "(overlaid, normalized)",
    y=1.03,
)

fig.savefig(
    os.path.join(
        GAIT_OUTPUT_DIR,
        "gait_heatmap_overlay_all_directions.png",
    ),
    dpi=200,
    bbox_inches="tight",
)

plt.show()
